<a href="https://colab.research.google.com/github/edgardlt03/ICO-Trabajos/blob/main/Feed_Forward_Network_con_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sympy==1.13.3 -q

In [3]:
import torch


print(f"PyTorch: {torch.__version__}")

# Usar GPU si no CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")

# random
torch.manual_seed(42)

PyTorch: 2.10.0+cpu
Dispositivo: cpu


In [1]:
from torchvision import datasets, transforms
#mirando su presentacion no supe como hacer el slpit sin numpy, asi que le pedi a la AI que cual era la mejor opcion y esta me dijo  usara esta libreria
from torch.utils.data import random_split

transform = transforms.Compose([
    transforms.ToTensor(),
])

full_train = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds    = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_ds, val_ds = random_split(full_train, [50000, 10000])

#esto tambien fue idea de la AI
print(f"Train:      {len(train_ds)}")
print(f"Validación: {len(val_ds)}")
print(f"Test:       {len(test_ds)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 57.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.64MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.47MB/s]

Train:      50000
Validación: 10000
Test:       10000


In [4]:
import torch.nn as nn
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(784, 128)
        self.relu   = nn.ReLU()
        self.output = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

model = MLP().to(device)
print(model)

MLP(
  (hidden): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=128, out_features=10, bias=True)
)


In [6]:
import torch.optim as optim
#CrossEntropy
loss_fn = nn.CrossEntropyLoss()

# Optimizador
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [8]:
model.train()

num_epochs = 10

for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)


        predictions = model(X_batch)
        loss = loss_fn(predictions, y_batch)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    model.eval()
    with torch.no_grad():
        val_loss = 0
        correct = 0
        total = 0
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            predictions = model(X_batch)
            val_loss += loss_fn(predictions, y_batch).item()
            correct += (predictions.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)

    print(f"Epoca {epoch+1}/{num_epochs} - Val Loss: {val_loss/total:.4f} - Val Acc: {correct/total:.4f}")
    model.train()

Época 1/10 - Val Loss: 0.0146 - Val Acc: 0.8778
Época 2/10 - Val Loss: 0.0113 - Val Acc: 0.8977
Época 3/10 - Val Loss: 0.0102 - Val Acc: 0.9056
Época 4/10 - Val Loss: 0.0095 - Val Acc: 0.9135
Época 5/10 - Val Loss: 0.0088 - Val Acc: 0.9196
Época 6/10 - Val Loss: 0.0083 - Val Acc: 0.9248
Época 7/10 - Val Loss: 0.0078 - Val Acc: 0.9286
Época 8/10 - Val Loss: 0.0075 - Val Acc: 0.9311
Época 9/10 - Val Loss: 0.0071 - Val Acc: 0.9360
Época 10/10 - Val Loss: 0.0068 - Val Acc: 0.9392
